In [ ]:
!pip install ultralytics --quiet

In [ ]:
import os, shutil, random, glob
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch

In [ ]:
from pathlib import Path

# Kaggle input datasets (you must add these in "Add Data")
FRACATLAS = Path("/kaggle/input/datasets/abdohamdg/fracatlas-dataset")
GRAZ_ROOT = Path("/kaggle/input/datasets/jasonroggy/grazpedwri-dx")

# Working directory
MERGED = Path("/kaggle/working/merged_fracture")

GRAZ_FRACTURE_CLASS_ID = 3

In [ ]:
from pathlib import Path

IMG_DIR = []
for i in range(1, 5):
    IMG_DIR += list(Path(f"/kaggle/input/datasets/jasonroggy/grazpedwri-dx/images_part{i}").glob("*.png"))

print("Total images:", len(IMG_DIR))

In [ ]:
LABEL_DIR = Path("/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels")

In [ ]:
valid_imgs = []

for img in IMG_DIR:
    lbl = LABEL_DIR / (img.stem + ".txt")
    if lbl.exists():
        valid_imgs.append(img)

print("Valid image-label pairs:", len(valid_imgs))

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(valid_imgs, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

graz_splits = {
    "train": train,
    "validation": val,
    "test": test
}

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
def find_split_dirs(root):
    """Auto-detect image/label dirs for each split."""
    splits = {}
    for split_name in ["train", "valid", "validation", "test"]:
        img_dir = None
        lbl_dir = None
        # Common structures
        for img_candidate in [
            root / "images" / split_name,
            root / split_name / "images",
            root / "data" / "images" / split_name,
        ]:
            if img_candidate.exists():
                img_dir = img_candidate
                break
        for lbl_candidate in [
            root / "labels" / split_name,
            root / split_name / "labels",
            root / "data" / "labels" / split_name,
        ]:
            if lbl_candidate.exists():
                lbl_dir = lbl_candidate
                break
        if img_dir and lbl_dir:
            # normalize split name
            key = "validation" if split_name == "valid" else split_name
            splits[key] = (img_dir, lbl_dir)
    return splits

IMG_DIR = []
for i in range(1, 5):
    IMG_DIR += list(Path(f"/kaggle/input/datasets/jasonroggy/grazpedwri-dx/images_part{i}").glob("*.png"))

print("Total images:", len(IMG_DIR))

# keep only images with labels
LABEL_DIR = Path("/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels")

valid_imgs = []
for img in IMG_DIR:
    if (LABEL_DIR / (img.stem + ".txt")).exists():
        valid_imgs.append(img)

print("Valid pairs:", len(valid_imgs))

# split
train, temp = train_test_split(valid_imgs, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

graz_splits = {
    "train": train,
    "validation": val,
    "test": test
}

In [ ]:
print("\n── Building merged dataset ──")

for split in ["train", "validation", "test"]:
    (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)

In [ ]:
def copy_fracatlas(src_root, split_map):
    """Copy FracAtlas — labels already nc=1, no remapping needed."""
    copied = 0
    for src_split, dst_split in split_map.items():
        img_dir = src_root / src_split / "images"
        lbl_dir = src_root / src_split / "labels"
        if not img_dir.exists():
            print(f"  ⚠ FracAtlas {src_split}/images not found, skipping")
            continue
        for img in img_dir.glob("*.jpg"):
            dst_img = MERGED / dst_split / "images" / f"FA_{img.name}"
            shutil.copy2(img, dst_img)
            lbl = lbl_dir / (img.stem + ".txt")
            dst_lbl = MERGED / dst_split / "labels" / f"FA_{img.stem}.txt"
            if lbl.exists():
                shutil.copy2(lbl, dst_lbl)
            else:
                dst_lbl.touch()   # empty = no fracture
            copied += 1
    print(f"  FracAtlas  → {copied} images copied")

In [ ]:
def remap_and_copy_graz(graz_splits):
    copied = total_boxes = 0

    for split, img_list in graz_splits.items():
        for img in img_list:
            dst_img = MERGED / split / "images" / f"GR_{img.name}"
            shutil.copy2(img, dst_img)

            lbl = LABEL_DIR / (img.stem + ".txt")
            dst_lbl = MERGED / split / "labels" / f"GR_{img.stem}.txt"

            fracture_lines = []

            if lbl.exists():
                for line in open(lbl).readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls = int(parts[0])

                        # KEEP ONLY fracture class
                        if cls == GRAZ_FRACTURE_CLASS_ID:
                            fracture_lines.append(
                                f"0 {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n"
                            )

            with open(dst_lbl, "w") as f:
                f.writelines(fracture_lines)

            total_boxes += len(fracture_lines)
            copied += 1

    print(f"✅ GRAZ → {copied} images | {total_boxes} fracture boxes")

In [ ]:
import os, shutil
from pathlib import Path

FRACATLAS = Path("/kaggle/input/datasets/abdohamdg/fracatlas-dataset/Fracatlas")

# ── Fix the typo: lables → labels ──────────────────────────
val_labels = FRACATLAS / "validation" / "labels"

if not val_labels.exists():
    val_labels = FRACATLAS / "validation" / "lables"  # fallback
# ── Write corrected data.yaml ───────────────────────────────
yaml_content = f"""path  : {FRACATLAS}
train : train/images
val   : validation/images
test  : test/images

nc    : 1
names : ["fractured"]
"""

yaml_path = Path("/kaggle/working/data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print("✅ data.yaml fixed")
print(yaml_content)

# ── Verify label/image pairing ──────────────────────────────
for split in ["train", "validation", "test"]:
    imgs   = list((FRACATLAS / split / "images").glob("*.jpg"))
    labels = list((FRACATLAS / split / "labels").glob("*.txt"))
    print(f"{split:12s} → images: {len(imgs):4d} | labels: {len(labels):4d}")

# ── Quick visual check — 6 random samples with boxes ───────
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
samples   = random.sample(list((FRACATLAS / "train" / "images").glob("*.jpg")), 6)

for ax, img_path in zip(axes.flatten(), samples):
    lbl_path = FRACATLAS / "train" / "labels" / (img_path.stem + ".txt")
    img      = Image.open(img_path).convert("RGB")
    w, h     = img.size
    ax.imshow(img)

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, cx, cy, nw, nh = map(float, parts)
                    x1 = (cx - nw/2) * w
                    y1 = (cy - nh/2) * h
                    rect = patches.Rectangle(
                        (x1, y1), nw*w, nh*h,
                        linewidth=3, edgecolor="red", facecolor="none"
                    )
                    ax.add_patch(rect)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")

plt.suptitle("FracAtlas — Verification (red = fracture box)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fracatlas_verify.png", dpi=120)
plt.show()
print("✅ Part 1 complete — proceed to Part 2")

In [ ]:
copy_fracatlas(FRACATLAS, {
    "train"     : "train",
    "validation": "validation",
    "test"      : "test",
})


In [ ]:
remap_and_copy_graz(graz_splits)

In [ ]:
print("\nMerged dataset summary:")
for split in ["train", "validation", "test"]:
    imgs = list((MERGED / split / "images").glob("*"))
    lbls = list((MERGED / split / "labels").glob("*.txt"))
    boxes = sum(
        sum(1 for l in open(f).readlines() if l.strip())
        for f in (MERGED / split / "labels").glob("*.txt")
    )
    print(f"  {split:12s} → images: {len(imgs):5d} | labels: {len(lbls):5d} | fracture boxes: {boxes:6d}")



In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR   = Path("/kaggle/working")

In [ ]:
def verify_dataset(n=8):
    train_imgs = list((MERGED / "train" / "images").glob("*"))
    samples    = random.sample(train_imgs, min(n, len(train_imgs)))

    fig, axes  = plt.subplots(2, 4, figsize=(20, 10))
    for ax, img_path in zip(axes.flatten(), samples):
        lbl_path = MERGED / "train" / "labels" / (img_path.stem + ".txt")
        try:
            img  = Image.open(img_path).convert("RGB")
        except Exception:
            ax.axis("off"); continue
        w, h = img.size
        ax.imshow(img)
        n_boxes = 0
        if lbl_path.exists():
            for line in open(lbl_path):
                parts = line.strip().split()
                if len(parts) == 5:
                    _, cx, cy, nw, nh = map(float, parts)
                    x1 = (cx-nw/2)*w; y1 = (cy-nh/2)*h
                    rect = patches.Rectangle(
                        (x1,y1), nw*w, nh*h,
                        linewidth=3, edgecolor="red", facecolor="none"
                    )
                    ax.add_patch(rect)
                    n_boxes += 1
        source = "FA" if img_path.name.startswith("FA_") else "GR"
        ax.set_title(f"[{source}] {n_boxes} box(es)", fontsize=9)
        ax.axis("off")

    plt.suptitle("Dataset Verification — red=fracture box",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "verify.png", dpi=100)
    plt.show()
    print("✅ Verification plot saved")

verify_dataset()

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/fracatlas-dataset/Fracatlas/validation/lables")
DST = Path("/kaggle/working/fixed_fracatlas/validation/labels")

DST.mkdir(parents=True, exist_ok=True)

for file in SRC.glob("*.txt"):
    shutil.copy2(file, DST / file.name)

print("✅ Fixed validation labels")

In [ ]:
IMG_SRC = Path("/kaggle/input/fracatlas-dataset/Fracatlas/validation/images")
IMG_DST = Path("/kaggle/working/fixed_fracatlas/validation/images")

IMG_DST.mkdir(parents=True, exist_ok=True)

for file in IMG_SRC.glob("*.jpg"):
    shutil.copy2(file, IMG_DST / file.name)

In [ ]:
from pathlib import Path

yaml_path = Path("/kaggle/working/data.yaml")

yaml_content = """
path: /kaggle/working/merged_fracture

train: train/images
val: validation/images
test: test/images

nc: 1
names: ["fractured"]
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("✅ YAML fixed")

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/merged_fracture")

print((base / "train/images").exists())
print((base / "validation/images").exists())
print((base / "test/images").exists())

In [ ]:
print("\n" + "="*60)
print("Training YOLOv8m — Kaggle T4")
print("="*60)

model = YOLO("yolov8m.pt")

results = model.train(
    data          = str(yaml_path),

    epochs        = 80,
    imgsz         = 640,
    batch         = 8,            # safer for T4

    lr0           = 1e-3,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 5e-4,
    warmup_epochs = 3,
    patience      = 15,

    project       = str(OUTPUT_DIR / "runs"),
    name          = "fracture_v1",

    device        = 0,
    workers       = 2,            # more stable than 4
    seed          = 42,
    exist_ok      = True,
    pretrained    = True,
    optimizer     = "AdamW",

    cos_lr        = True,
    close_mosaic  = 20,
    amp           = True,

    val           = True,
    plots         = True,
    save_period   = 10,

    # X-ray augmentation (KEEP — important)
    fliplr        = 0.5,
    flipud        = 0.1,
    degrees       = 20.0,
    translate     = 0.1,
    scale         = 0.5,
    shear         = 2.0,
    hsv_h         = 0.0,
    hsv_s         = 0.0,
    hsv_v         = 0.3,
    mosaic        = 1.0,
    mixup         = 0.3,
    copy_paste    = 0.0,

    freeze        = 10,   # 🔥 added
)

BEST_MODEL = OUTPUT_DIR / "runs" / "fracture_v1" / "weights" / "best.pt"

print("\n✅ Training complete")
print(f"Best model: {BEST_MODEL}")

In [1]:
print("\n" + "="*60)
print("Threshold Sweep — Test Set")
print("="*60)

best_model  = YOLO(str(BEST_MODEL))
best_conf   = 0.25
best_recall = 0.0
results_table = []

print(f"\n{'conf':>6} | {'Precision':>10} | {'Recall':>8} | {'mAP50':>8} | {'mAP50-95':>10}")
print("-" * 55)

for conf in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    m   = best_model.val(
        data    = str(yaml_path),
        split   = "test",
        imgsz   = 640,
        conf    = conf,
        iou     = 0.45,
        verbose = False,
    )
    p, r, map50, map5095 = m.box.mp, m.box.mr, m.box.map50, m.box.map
    print(f"{conf:>6.2f} | {p:>10.3f} | {r:>8.3f} | {map50:>8.3f} | {map5095:>10.3f}")
    results_table.append((conf, p, r, map50, map5095))
    if r > best_recall:
        best_recall = r
        best_conf   = conf

print(f"\n✅ Best operating point → conf={best_conf} | recall={best_recall:.3f}")


Threshold Sweep — Test Set


NameError: name 'YOLO' is not defined

In [ ]:
print(f"\nFinal evaluation at conf={best_conf}:")
best_model.val(
    data    = str(yaml_path),
    split   = "test",
    imgsz   = 640,
    conf    = best_conf,
    iou     = 0.45,
    verbose = True,
    plots   = True,
)


In [ ]:
def visualize_predictions(model, conf, n=8,
                           save_path="predictions.png"):
    test_imgs = list((MERGED / "test" / "images").glob("*"))
    chosen    = random.sample(test_imgs, min(n, len(test_imgs)))

    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5))
    axes = axes.flatten()

    for i, img_path in enumerate(chosen):
        res      = model.predict(img_path, conf=conf, verbose=False)
        result   = res[0]
        lbl_path = MERGED / "test" / "labels" / (img_path.stem + ".txt")
        source   = "FracAtlas" if img_path.name.startswith("FA_") else "GRAZ"

        img    = Image.open(img_path).convert("RGB")
        img_np = np.array(img)
        h, w   = img_np.shape[:2]
        axes[i].imshow(img_np, cmap="gray")

        # GT — blue dashed
        if lbl_path.exists():
            for line in open(lbl_path):
                parts = line.strip().split()
                if len(parts) == 5:
                    _, cx, cy, nw, nh = map(float, parts)
                    x1 = (cx-nw/2)*w; y1 = (cy-nh/2)*h
                    rect = patches.Rectangle(
                        (x1,y1), nw*w, nh*h,
                        linewidth=2, edgecolor="blue",
                        facecolor="none", linestyle="--"
                    )
                    axes[i].add_patch(rect)

        # Predictions — red
        n_det = 0
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
                cf           = float(box.conf[0].cpu())
                rect = patches.Rectangle(
                    (x1,y1), x2-x1, y2-y1,
                    linewidth=2, edgecolor="red", facecolor="none"
                )
                axes[i].add_patch(rect)
                axes[i].text(
                    x1, max(y1-5,0), f"{cf:.2f}",
                    color="red", fontsize=9, fontweight="bold"
                )
                n_det += 1

        axes[i].set_title(f"[{source}] {n_det} pred", fontsize=9)
        axes[i].axis("off")

    for j in range(len(chosen), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"YOLOv8l Predictions — Blue=GT, Red=Pred (conf={conf})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")

In [ ]:
visualize_predictions(best_model, conf=best_conf)

In [ ]:
import shutil
shutil.copy2(
    "/kaggle/working/runs/detect/fracture_v1/weights/best.pt",
    "/kaggle/working/best_yolo_fracture.pt"
)

In [2]:
from ultralytics import YOLO

MODEL_PATH = "/kaggle/working/runs/fracture_v1/weights/best.pt"

model = YOLO(MODEL_PATH)

print("✅ Model loaded")

✅ Model loaded


In [3]:
from pathlib import Path

MERGED = Path("/kaggle/working/merged_fracture")

yaml_path = "/kaggle/working/data.yaml"

with open(yaml_path, "w") as f:
    f.write(f"""
path: {MERGED}
train: train/images
val: validation/images
test: test/images

nc: 1
names: ["fractured"]
""")

print("✅ YAML ready")

✅ YAML ready


In [4]:
metrics = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    conf=0.25,
    iou=0.45
)

print(metrics.results_dict)

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 81.7±31.1 MB/s, size: 766.6 KB)
val: Scanning /kaggle/working/merged_fracture/test/labels... 2094 images, 682 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2094/2094 209.7it/s 10.0s.1s
val: New cache created: /kaggle/working/merged_fracture/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 131/131 4.2it/s 31.5s0.2s
                   all       2094       1888      0.913      0.871      0.922      0.569
Speed: 0.6ms preprocess, 12.0ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
{'metrics/precision(B)': 0.9132005605210778, 'metrics/recall(B)': 0.871292372881356, 'metrics/mAP50(B)': 0.9223267551129288, 'metrics/mAP50-95(B)': 0.5692351825678188, 'fitnes

In [5]:
best_conf = 0.25
best_recall = 0.0

for conf in [0.10, 0.15, 0.20, 0.25, 0.30]:
    m = model.val(
        data=yaml_path,
        split="test",
        imgsz=640,
        conf=conf,
        iou=0.45,
        verbose=False
    )

    p, r = m.box.mp, m.box.mr
    print(f"conf={conf:.2f} → precision={p:.3f}, recall={r:.3f}")

    if r > best_recall:
        best_recall = r
        best_conf = conf

print(f"\n✅ Best conf: {best_conf} (recall={best_recall:.3f})")

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2658.7±1412.0 MB/s, size: 833.4 KB)
val: Scanning /kaggle/working/merged_fracture/test/labels.cache... 2094 images, 682 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2094/2094 975.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 131/131 3.9it/s 33.6s0.2s
                   all       2094       1888      0.917       0.87      0.933      0.567
Speed: 0.6ms preprocess, 12.8ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2
conf=0.10 → precision=0.917, recall=0.870
Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1989.7±215.4 MB/s, size: 863.8 KB)
val: Scanning /kaggle/working/merged_fracture/test/labels.cache... 2094 images, 682 backgrounds, 0 corrupt: 100% 

In [6]:
results = model.predict(
    source=str(MERGED / "test/images"),
    conf=best_conf,
    save=True
)

print("✅ Predictions saved in /kaggle/working/runs/")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2094 /kaggle/working/merged_fracture/test/images/FA_IMG0003297.jpg: 640x544 1 fractured, 59.7ms
image 2/2094 /kaggle/working/merged_fracture/test/images/FA_IMG0003298.jpg: 640x544 2 fractureds, 34.9ms
image 3/2094 /kaggle/working/merged_fracture/test/images/FA_IMG0003301.jpg: 640x544 (no detections), 34.9ms
image 4/2094 /kaggle/working/merged_fracture/test/images/FA_IMG0003308.jpg: 640x544 1 fractured, 34.9ms
image 5/2094 /kaggle/working/merged_f